# Verifying the Outputs of the PHYSys

## 1. Generating BPSK Sweeps across -10 to +10 dBs

To generate the dataset, let us use the following command per the documentation:

> python -m physys.cli sweep -c modulations/BPSK/config.json -o modulations/BPSK/sweep.h5

Used sites:

1. [h5py - Quick Start Guide](https://docs.h5py.org/en/latest/quick.html)

### Imports and File Directories

In [106]:
import h5py
import matplotlib.pyplot as plt
import numpy as np

H5_DIR = './modulations/BPSK'
H5_FILE = H5_DIR + '/sweep.h5'

### Inspection of Metadata and File Structure

In [107]:

h5_file = h5py.File(H5_FILE, 'r')

KEYS = list(h5_file.keys())

print(KEYS)


['bits', 'llr', 'snr_labels']


**FINDING #1**: Key inspection revealed a crucial detail about how I wrote the PHYSys the ebno_db are not listed throughly I need to check this. As the output on version 1 before the change on export.py looks like this:

``` text
['ebno_db=-10.0000',
 'ebno_db=-2.0000',
 'ebno_db=-4.0000',
 'ebno_db=-6.0000',
 'ebno_db=-8.0000',
 'ebno_db=0.0000',
 'ebno_db=10.0000',
 'ebno_db=2.0000',
 'ebno_db=4.0000',
 'ebno_db=6.0000',
 'ebno_db=8.0000']
```

which changed to above versino which is:

````
['bits', 'llr', 'snr_labels']
````

In [108]:
bits_name       = KEYS[0]
llr_name        = KEYS[1]
snr_labels_name = KEYS[2]

BITS_DATASET       = h5_file[bits_name]
LLR_DATASET        = h5_file[llr_name]
SNR_LABELS_DATASET = h5_file[snr_labels_name]

bits            = BITS_DATASET[...]
llrs            = LLR_DATASET[...]
snrs            = SNR_LABELS_DATASET[...]

print(f"Bits shape: {snrs.shape}")
print(f"LLRs shape: {snrs.shape}")
print(f"SNRs shape: {snrs.shape}")


print(f"Bits dtype: {bits.dtype}")
print(f"First element of bits: {bits[0]}")
print(f"First element's shape  of bits: {bits[0].shape}")

Bits shape: (11,)
LLRs shape: (11,)
SNRs shape: (11,)
Bits dtype: float32
First element of bits: [[1. 1. 1. ... 0. 0. 0.]
 [0. 1. 0. ... 1. 0. 1.]
 [1. 1. 0. ... 0. 0. 1.]
 [1. 1. 1. ... 0. 0. 0.]]
First element's shape  of bits: (4, 1024)


**FINDING #2** The shape of each key's corresponding dataset showcases the exact shape of
(11,) which means it is on a lower level in the hierarchy.

In [109]:
def inspect_structure(name, node):

    if isinstance(node, h5py.Dataset):
        print(f"Dataset: {name:<12} | Shape: {str(node.shape):<15} | Type: {node.dtype}")
    
    elif isinstance(node, h5py.Group):
        print(f"    Group: {name}")

h5_file.visititems(inspect_structure)

Dataset: bits         | Shape: (11, 4, 1024)   | Type: float32
Dataset: llr          | Shape: (11, 4, 1024)   | Type: float32
Dataset: snr_labels   | Shape: (11,)           | Type: float64


In [110]:
print(f"SNR Values: {snrs}")

SNR Values: [-10.  -8.  -6.  -4.  -2.   0.   2.   4.   6.   8.  10.]


**CONCLUSION 1** The old tree structure followed

```text
ebno_db=-10.0000
    bits (Shape: 4, 1024)
    llr  (Shape: 4, 1024)
ebno_db=-8.0000
    bits (Shape: 4, 1024)
    ...
```

The new structure follows exactly:

```text
bits       (Shape: 11, 4, 1024)
llr        (Shape: 11, 4, 1024)
snr_labels (Shape: 11,)
```

Checking the attributes **metadata**

In [111]:
FILE_ATTRS = h5_file.attrs
FILE_ATTRS_KEYS = list(FILE_ATTRS.keys())

print("File attributes keys: ", FILE_ATTRS_KEYS)

print("Created at: ", FILE_ATTRS.get('created_utc'))

print(FILE_ATTRS.get('config_json'))

File attributes keys:  ['config_json', 'created_utc']
Created at:  2026-08-05T14:46:26Z
{"common": {"precision": "single", "device": null, "active_channel": "awgn", "active_waveform": "time"}, "source": {}, "modulation": {"type": "pam", "num_bits_per_symbol": 1, "mapper": {"return_indices": false}, "demapper": {"hard_out": false, "demapping_method": "app"}, "constellation": {"normalize": false, "center": false, "points": null}}, "channels": {"awgn": {"precision": null, "device": null}, "tdl": {"model": "A", "delay_spread": 1e-07, "carrier_frequency": 3500000000.0, "num_sinusoids": 20, "los_angle_of_arrival": 0.7853981633974483, "min_speed": 0.0, "max_speed": null, "num_rx_ant": 1, "num_tx_ant": 1, "spatial_corr_mat": null, "rx_corr_mat": null, "tx_corr_mat": null, "precision": null, "device": null}, "system_level": {"variant": "umi", "carrier_frequency": 140000000000.0, "o2i_model": "low", "direction": "downlink", "enable_pathloss": true, "enable_shadow_fading": true, "bs_array": {"num

### Sanity Checks

#### 1. Checking for NaN and Inf

In [112]:
print( "----- Checking for NaN and Inf -----" )

print("--- BITS")
print(  np.isnan(bits).any()  )
print(  np.isinf(bits).any()  )

print("--- LLRS")
print(  np.isnan(llrs).any()  )
print(  np.isinf(llrs).any()  )

print("--- SNRS")
print(  np.isnan(snrs).any()  )
print(  np.isinf(snrs).any()  )

----- Checking for NaN and Inf -----
--- BITS
False
False
--- LLRS
False
False
--- SNRS
False
False


#### 2. Bits Distribution

In [113]:
print(np.mean(bits))

0.50337356


#### 3. LLR Magnitude over SNR 

In [120]:
print("SNR Levels: ", snrs)

LOWEST_LLR   = llrs[0]
HIGHEST_LLR  = llrs[-1]

LOWEST_SNR   = snrs[0]
HIGHEST_SNR  = snrs[-1]

AVG_LOWEST_LLR  = np.mean(abs(LOWEST_LLR))
AVG_HIGHEST_LLR = np.mean(abs(HIGHEST_LLR))

print("Average LLR at lowest SNR: ")
print(  AVG_LOWEST_LLR  )

print("Average LLR at highest SNR: ")
print(  AVG_HIGHEST_LLR  )

SNR Levels:  [-10.  -8.  -6.  -4.  -2.   0.   2.   4.   6.   8.  10.]
Average LLR at lowest SNR: 
0.78955495
Average LLR at highest SNR: 
40.06988


#### 4. Bits vs LLR Sign

In [116]:
HIGHEST_BITS = bits[-1]

LLRS_FOR_ZEROS = HIGHEST_LLR[HIGHEST_BITS == 0]
LLRS_FOR_ONES  = HIGHEST_LLR[HIGHEST_BITS == 1]

MEAN_FOR_ZEROS = np.mean(LLRS_FOR_ZEROS)
MEAN_FOR_ONES  = np.mean(LLRS_FOR_ONES)

print("Mean LLR when bit is 0: ", MEAN_FOR_ZEROS)
print("Mean LLR when bit is 1: ", MEAN_FOR_ONES)

Mean LLR when bit is 0:  -39.90882
Mean LLR when bit is 1:  40.23687


**FINDING #3** The output of the above code when config.json's active_channel is set to system_level is below:

```text
Mean LLR when bit is 0:  0.09027589
Mean LLR when bit is 1:  0.11050552
```

When the active_channel is changed to awgn, the output changes to:

```text
Mean LLR when bit is 0:  -39.90882
Mean LLR when bit is 1:  40.23687
```

During the final numerical sanity check, we observed that the LLRs generated under the system_level (UMi/UMa) fading channel were near zero (mean ~0.1), representing extreme uncertainty. However, when switching the channel to pure awgn, the LLRs instantly corrected themselves to massive confidence values (~40.0).

**CONCLUSION**

The current system does not favor the ultimate goal of making an AMR dataset.

Instead of deleting the working bits and llr pipeline, PHYSys will be **extended**. 

I will update runtime.py and export.py to optionally return and serialize the raw transmitted symbols (`x`) and the raw received noisy symbols (`y`). 

* Saving `y` provides the exact input features (`X`) needed for the PyTorch AMC models.
* Saving `x` allows to visualize clean vs. noisy constellation scatter plots.
* Keeping the `llr` pipeline preserves the codebase for future BER testing if an Equalizer is added later.



**SWITCHING** It makes more sense to switch to a different .ipynb and keep the current outputs and progression after the big I/Q change.